# BrandSphere AI — Week 3: Font Recommendation Engine (KNN Classifier)
**CRS AI Capstone 2025–26**

**Dataset:** `datasets/raw/font_dataset.csv` — 31 real font families with typographic feature vectors
(weight, contrast, letter-spacing, formality, energy), augmented to 465 samples.

Covers:
1. Font dataset exploration
2. Feature engineering (typographic properties)
3. KNN classifier training — Category (91.4% accuracy)
4. KNN classifier training — Personality mapping
5. Brand personality → font recommendation
6. Integration with logo and palette modules

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn 2>/dev/null
import subprocess, os, sys
r = subprocess.run(['git','clone','--quiet','https://github.com/iblamehemer/og','/content/BrandSphere'],capture_output=True,text=True)
os.chdir('/content/BrandSphere'); sys.path.insert(0,'/content/BrandSphere')
os.makedirs('assets/sample_exports',exist_ok=True)
print('✅ Ready')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams.update({'figure.facecolor':'#0C0D0F','axes.facecolor':'#141518','text.color':'white','axes.labelcolor':'white','xtick.color':'white','ytick.color':'white'})
import seaborn as sns
import pickle, warnings; warnings.filterwarnings('ignore')
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
print('✅ Libraries loaded')

## 1. Load Font Dataset

In [ ]:
df = pd.read_csv('datasets/raw/font_dataset.csv')
print(f'Font dataset: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
print(f'\nCategories: {df["category"].value_counts().to_dict()}')
print(f'\nPersonalities: {df["personality"].value_counts().to_dict()}')
print(f'\nSample fonts:')
df[['font_name','category','personality','weight','contrast','formality','energy']].head(10)

## 2. Feature Visualisation

In [ ]:
feature_cols = ['weight','contrast','spacing','formality','energy']
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Font Feature Distributions by Category', color='#C9A84C', fontsize=13)

cat_colors = {'Serif':'#C9A84C','Sans-Serif':'#3ECFB2','Display':'#F06292','Monospace':'#7B8FF7','Script':'#FFB74D'}
for i, (ax, feat) in enumerate(zip(axes, ['formality','energy'])):
    for cat, color in cat_colors.items():
        subset = df[df['category']==cat][feat]
        if len(subset) > 0:
            ax.hist(subset, bins=8, color=color, alpha=0.6, label=cat, edgecolor='none')
    ax.set_title(feat.capitalize(), color='white', fontsize=11)
    ax.set_xlabel(feat, color='white')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('assets/sample_exports/07_font_features.png', dpi=120, bbox_inches='tight', facecolor='#0C0D0F')
plt.show()

## 3. Augmented Dataset & Feature Engineering

In [ ]:
df_aug = pd.read_csv('datasets/processed/font_features.csv')
print(f'Augmented dataset: {df_aug.shape} ({len(df_aug)//len(df)}x per font)')

feature_cols = ['weight','contrast','spacing','formality','energy']
X = df_aug[feature_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Feature matrix: {X_scaled.shape}')
print(f'Features: {feature_cols}')
print(f'\nScaled feature statistics:')
scaled_df = pd.DataFrame(X_scaled, columns=feature_cols)
print(scaled_df.describe().round(3))

## 4. Train KNN Font Category Classifier

In [ ]:
le_cat = LabelEncoder()
y_cat  = le_cat.fit_transform(df_aug['category'])
print(f'Classes: {list(le_cat.classes_)}')

X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y_cat, test_size=0.2, random_state=42, stratify=y_cat)

models = {
    'KNN (k=3)':    KNeighborsClassifier(n_neighbors=3, metric='euclidean'),
    'KNN (k=5)':    KNeighborsClassifier(n_neighbors=5, metric='euclidean'),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
}

best_acc, best_m, best_name = 0, None, ""
for name, m in models.items():
    m.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, m.predict(X_te))
    cv  = cross_val_score(m, X_scaled, y_cat, cv=5).mean()
    print(f'  {name:15s}: Test Acc={acc:.1%}  CV={cv:.1%}')
    if acc > best_acc:
        best_acc, best_m, best_name = acc, m, name

print(f'\n✅ Best: {best_name} (Test Accuracy: {best_acc:.1%})')
print(f'Baseline (random): {1/len(le_cat.classes_):.1%}')
print(f'Improvement: {best_acc/(1/len(le_cat.classes_)):.1f}× better than random')

## 5. Confusion Matrix

In [ ]:
y_pred = best_m.predict(X_te)
cm = confusion_matrix(y_te, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=le_cat.classes_, yticklabels=le_cat.classes_, ax=ax)
ax.set_title(f'Font Category Confusion Matrix ({best_name})', color='#C9A84C', fontsize=12)
ax.set_xlabel('Predicted', color='white')
ax.set_ylabel('Actual', color='white')
plt.xticks(rotation=25, color='white')
plt.yticks(rotation=0, color='white')
plt.tight_layout()
plt.savefig('assets/sample_exports/07_confusion_matrix.png', dpi=120, bbox_inches='tight', facecolor='#141518')
plt.show()

print('\nClassification Report:')
print(classification_report(y_te, y_pred, target_names=le_cat.classes_))

## 6. Train Personality Classifier

In [ ]:
le_pers = LabelEncoder()
y_pers  = le_pers.fit_transform(df_aug['personality'])
print(f'Personality classes: {list(le_pers.classes_)}')

X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_scaled, y_pers, test_size=0.2, random_state=42, stratify=y_pers)
m_pers = RandomForestClassifier(n_estimators=100, random_state=42)
m_pers.fit(X_tr2, y_tr2)
acc_pers = accuracy_score(y_te2, m_pers.predict(X_te2))
cv_pers  = cross_val_score(m_pers, X_scaled, y_pers, cv=5).mean()
print(f'Personality classifier: Acc={acc_pers:.1%}  CV={cv_pers:.1%}')

## 7. Save Models

In [ ]:
import os
os.makedirs('models', exist_ok=True)

with open('models/font_category_classifier.pkl','wb') as f: pickle.dump(best_m, f)
with open('models/font_personality_classifier.pkl','wb') as f: pickle.dump(m_pers, f)
with open('models/font_scaler.pkl','wb') as f: pickle.dump(scaler, f)
with open('models/font_label_encoders.pkl','wb') as f:
    pickle.dump({'category': le_cat, 'personality': le_pers}, f)

import json
results = {
    'category_classifier': best_name, 'category_accuracy': round(best_acc,4),
    'personality_classifier': 'RandomForest', 'personality_accuracy': round(acc_pers,4),
    'dataset': {'fonts': len(df), 'samples': len(df_aug), 'features': feature_cols}
}
with open('models/font_training_results.json','w') as f: json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))
print('\n✅ All font models saved')

## 8. Font Recommendation Demo

In [ ]:
from src.font_engine import recommend_fonts
from src.config import INDUSTRIES, PERSONALITIES

print('Font recommendations using trained KNN + expert mapping:')
print('─'*60)
for industry, personality in [
    ('Finance', 'Luxury'), ('Technology / Software', 'Minimalist'),
    ('Food & Beverage', 'Playful'), ('Healthcare', 'Professional'),
]:
    fonts = recommend_fonts(industry, personality)
    top   = fonts[0]
    print(f'{personality:14s} × {industry:30s}')
    print(f'  → {top["heading"]} + {top["body"]} | {top["rationale"][:55]}…')
    print()

## Summary

| Model | Task | Accuracy | Method |
|---|---|---|---|
| KNN (k=3, euclidean) | Font category (5 classes) | **91.4%** | Typographic feature vectors |
| Random Forest | Font personality (7 classes) | **63.4%** | Typographic feature vectors |
| Baseline (random) | Category | 20.0% | — |

**KNN achieves 4.6× better than random** for category classification.
Dataset: 31 real font families × 15 augmentations = 465 samples.
Features: weight, contrast, letter-spacing, formality score, energy score.